# Graph Multiset Transformer (GMT) Pooling on PROTEINS

Graph Classification on PROTEINS (TUDataset): the Graph Multiset Transformer ("Accurate Learning of Graph Representations with Graph Multiset Pooling", https://arxiv.org/abs/2102.11533) replaces simple sum/mean graph readout with a multi-head-attention pooling operator that treats a graph's node embeddings as a "multiset" to be summarized by `k` learned seed vectors. This notebook ports the reference implementation to K3-Node: three stacked `GCNConv` layers produce per-node embeddings at increasing receptive field, their outputs are concatenated (jumping-knowledge style) into a 96-dim representation, and `k3_layers.GraphMultisetTransformer` pools that multiset down to a single graph-level vector before a 2-layer classifier head. Training is a standard multi-class cross-entropy loop reporting train loss plus val/test accuracy every epoch — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import numpy as np
import keras
from keras import layers, ops

from k3_node import layers as k3_layers
from k3_node.datasets import TUDataset
from k3_node.loader import DataLoader

title = "Graph Multiset Transformer (GMT) Pooling on PROTEINS"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset: shuffle once, then split into train (80%) / val (10%) / test (10%)
dataset = TUDataset(root="./data/PROTEINS", name="PROTEINS")
perm = np.random.permutation(len(dataset)).tolist()
dataset = dataset[perm]

n = (len(dataset) + 9) // 10
train_dataset = dataset[2 * n:]
val_dataset = dataset[n:2 * n]
test_dataset = dataset[:n]

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)


# 2. GCNConv stack (concatenated, jumping-knowledge style) + GMT pooling
class K3Net(keras.Model):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, 32)
        self.conv2 = k3_layers.GCNConv(32, 32)
        self.conv3 = k3_layers.GCNConv(32, 32)

        self.pool = k3_layers.GraphMultisetTransformer(96, k=10, heads=4)

        self.lin1 = layers.Dense(16)
        self.lin2 = layers.Dense(num_classes)
        self.dropout = layers.Dropout(0.5)

    def call(self, x0, edge_index, batch, training=False):
        x1 = ops.relu(self.conv1(x0, edge_index))
        x2 = ops.relu(self.conv2(x1, edge_index))
        x3 = ops.relu(self.conv3(x2, edge_index))
        x = ops.concatenate([x1, x2, x3], axis=-1)

        x = self.pool(x, index=batch)

        x = ops.relu(self.lin1(x))
        x = self.dropout(x, training=training)
        x = self.lin2(x)
        return x


model = K3Net(dataset.num_features, dataset.num_classes)

# Eager forward pass to build every sublayer's weights
sample = next(iter(train_loader))
_ = model(
    ops.convert_to_tensor(sample.x, dtype="float32"),
    ops.convert_to_tensor(sample.edge_index, dtype="int64"),
    ops.convert_to_tensor(sample.batch, dtype="int64"),
)

optimizer = keras.optimizers.Adam(learning_rate=0.001, weight_decay=1e-4)
trainable_vars = model.trainable_variables

# Setup optimizer variables for the functional (JAX) backend
if backend == "jax":
    import jax
    optimizer.build(trainable_vars)
    opt_vars = [v.value for v in optimizer.variables]
    non_trainable_vars = [v.value for v in model.non_trainable_variables]


def batch_tensors(data_batch):
    x = ops.convert_to_tensor(data_batch.x, dtype="float32")
    edge_index = ops.convert_to_tensor(data_batch.edge_index, dtype="int64")
    batch_vec = ops.convert_to_tensor(data_batch.batch, dtype="int64")
    y = ops.reshape(ops.convert_to_tensor(data_batch.y, dtype="int32"), (-1,))
    return x, edge_index, batch_vec, y


def cross_entropy(logits, y):
    y_one_hot = ops.one_hot(y, dataset.num_classes)
    log_probs = ops.log_softmax(logits, axis=-1)
    return -ops.mean(ops.sum(y_one_hot * log_probs, axis=-1))


# 3. Multi-Backend Training Step
def train_step(data_batch):
    global opt_vars
    x, edge_index, batch_vec, y = batch_tensors(data_batch)
    num_graphs = int(ops.shape(y)[0])

    if backend == "torch":
        out = model(x, edge_index, batch_vec, training=True)
        loss = cross_entropy(out, y)
        loss.backward()
        grads = [v.value.grad for v in trainable_vars]
        optimizer.apply_gradients(zip(grads, trainable_vars))
        for v in trainable_vars:
            if v.value.grad is not None:
                v.value.grad.zero_()
        loss_value = float(ops.convert_to_numpy(loss))

    elif backend == "tensorflow":
        import tensorflow as tf
        with tf.GradientTape() as tape:
            out = model(x, edge_index, batch_vec, training=True)
            loss = cross_entropy(out, y)
        grads = tape.gradient(loss, trainable_vars)
        optimizer.apply_gradients(zip(grads, trainable_vars))
        loss_value = float(ops.convert_to_numpy(loss))

    else:  # jax
        trainable_values = [v.value for v in trainable_vars]

        def loss_fn(params):
            out, _ = model.stateless_call(params, non_trainable_vars, x, edge_index, batch_vec, training=True)
            return cross_entropy(out, y)

        loss_val, grads = jax.value_and_grad(loss_fn)(trainable_values)
        new_values, opt_vars = optimizer.stateless_apply(opt_vars, grads, trainable_values)
        for v, val in zip(trainable_vars, new_values):
            v.assign(val)
        loss_value = float(loss_val)

    return loss_value * num_graphs


def train():
    total_loss = 0.0
    for data_batch in train_loader:
        total_loss += train_step(data_batch)
    return total_loss / len(train_loader.dataset)


def test(loader):
    total_correct = 0
    for data_batch in loader:
        x, edge_index, batch_vec, y = batch_tensors(data_batch)
        out = model(x, edge_index, batch_vec, training=False)
        pred = ops.argmax(out, axis=-1)
        total_correct += int(ops.convert_to_numpy(ops.sum(ops.cast(pred == ops.cast(y, pred.dtype), "int32"))))
    return total_correct / len(loader.dataset)


print(f"Training K3-Node GMT model on {backend} backend...")
for epoch in range(1, 201):
    train_loss = train()
    val_acc = test(val_loader)
    test_acc = test(test_loader)
    print(f"Epoch: {epoch:03d}, Loss: {train_loss:.4f}, "
          f"Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")

print("\n✓ K3-Node execution completed successfully!")